# ex05 · softmax 与交叉熵（对应教材 3.4 softmax回归）

> **做题流程**：每道题先手算/先判断，再运行代码验证。
> **做完再看** `solutions/ex05-答案.md`。
>
> 难度标记：🌱 基础（预测+验证）｜🔧 变式（改动观察）｜🚀 挑战（闭卷复现）
>
> 本节是面试高频密集区：softmax 数值稳定、交叉熵与极大似然的关系，一轮必问。

In [1]:
import torch
import math

def softmax(X):
    """朴素 softmax：exp -> 每行求和 -> 归一化"""
    X_exp = torch.exp(X)
    partition = X_exp.sum(1, keepdim=True)
    return X_exp / partition

## 题 1 🌱 手算 softmax([1, 2, 3])

三个 logits 是 [1, 2, 3]。先手算 softmax 概率（写成 e¹/(e¹+e²+e³) 的完整过程，保留 4 位小数），再运行验证。

**【你的预测】**

In [2]:
logits = torch.tensor([1.0, 2.0, 3.0])
probs = softmax(logits.reshape(1, -1))
p1 = probs.flatten()
print('softmax([1,2,3]) =', [round(x, 4) for x in p1.tolist()])
print('概率和 =', probs.sum().item())

softmax([1,2,3]) = [0.09, 0.2447, 0.6652]
概率和 = 1.0


## 题 2 🔧 数值稳定性：为什么要「减 max」

把 logits 换成 [1000, 1001, 1002]。先判断再运行：

- 直接算 exp(1000) 会得到什么？（float 最大约 3.4×10³⁸，exp(1000) 呢？）
- 每个 logit 先减去最大值再算 softmax，结果会变吗？为什么可以这么做？

**【你的预测】**
**为什么减常数不改变结果**：

softmax(xᵢ − c) = e^{xᵢ−c} / Σe^{xⱼ−c} = e⁻ᶜ·e^{xᵢ} / (e⁻ᶜ·Σe^{xⱼ}) = e^{xᵢ} / Σe^{xⱼ} = softmax(xᵢ)

分子分母的 e⁻ᶜ 约掉了。取 c = max(x) 可以让最大的指数项为 e⁰ = 1，其余都是 ≤1 的数，彻底避免上溢。这是面试高频考点「softmax 为什么减 max」。

In [4]:
big = torch.tensor([1000.0, 1001.0, 1002.0])
naive = softmax(big.reshape(1, -1)).flatten()
print('朴素 softmax([1000,1001,1002]) =', naive.tolist())

shifted = softmax((big - big.max()).reshape(1, -1)).flatten()
p2 = shifted
print('减 max 后 =', [round(x, 4) for x in shifted.tolist()])

# 小数值下验证「减常数不改变结果」
shifted_small = softmax((logits - logits.max()).reshape(1, -1)).flatten()
print('减 max 后与题 1 结果一致?', torch.allclose(shifted_small, p1, atol=1e-6))

朴素 softmax([1000,1001,1002]) = [nan, nan, nan]
减 max 后 = [0.09, 0.2447, 0.6652]
减 max 后与题 1 结果一致? True


## 题 3 🌱 手算交叉熵

真实标签是第 2 类（one-hot [0, 0, 1]），预测概率就是题 1 的结果 [0.0900, 0.2447, 0.6652]。
交叉熵 = 真实类预测概率的负对数。先手算 −ln(0.6652)，再运行验证。

**【你的预测】**

In [5]:
y_idx = 2
ce_val = -torch.log(p1[y_idx])
print('交叉熵 =', round(ce_val.item(), 4))

交叉熵 = 0.4076


## 题 4 🔧 交叉熵 = 负对数似然（面试高频）

一批 3 个样本：预测概率矩阵 y_hat（每行一个样本的完整分布）、真实标签 y = [2, 0, 1]。
先手算「每个样本正确类的预测概率」，取负对数再求平均，然后运行验证。
想清楚后写你的理解：为什么说「最小化交叉熵 = 最大化似然」？（面试必问）

**【你的预测】**
**为什么「最小化交叉熵 = 最大化似然」**：

- 似然 = 模型在这些样本上给出正确类别的概率的**乘积**：L = Πᵢ P(yᵢ | xᵢ)
- 取对数变求和（乘积转加法、数值稳定）：log L = Σᵢ log P(yᵢ|xᵢ)
- 最大化 log L ⇔ 最小化 −Σᵢ log P(yᵢ|xᵢ) ⇔ 最小化 Σᵢ CEᵢ（平均后即交叉熵损失）

所以训练时最小化交叉熵，就是在做极大似然估计——模型被训练成「最可能产生这批数据」的参数。这就是面试必问题「为什么最小化交叉熵 = 极大似然」的完整链条。

In [6]:
y_hat = torch.tensor([[0.1, 0.2, 0.7],
                      [0.8, 0.1, 0.1],
                      [0.2, 0.5, 0.3]])
y = torch.tensor([2, 0, 1])
# 用「真实标签作索引」取出每个样本正确类的概率
picked = y_hat[range(len(y_hat)), y]
ce_batch = (-torch.log(picked)).mean()
print('每个样本正确类的概率:', picked.tolist())
print('整批平均交叉熵 =', round(ce_batch.item(), 4))

每个样本正确类的概率: [0.699999988079071, 0.800000011920929, 0.5]
整批平均交叉熵 = 0.4243


## 题 5 🌱 softmax 的性质（先判断，不运行）

判断下面说法对错并写理由：

1. softmax 的输出都在 [0, 1] 内，且每行和为 1

e**x的非负性以及概率总和为1的约束性
2. 所有 logits 同时加同一个常数 c，softmax 输出不变

e**x的指数运算性， +c = *e^c，上下相除，得到的softmax输出不变
3. softmax 输出最大的位置，就是 logits 最大的位置

对，因为softmax运算的保序性（log单调性不变）
**【你的预测】**

In [7]:
x = torch.tensor([[-1.0, 0.5, 2.0], [3.0, 1.0, -2.0]])
p = softmax(x)
print('每行和:', p.sum(1).tolist())
print('每行最小值:', p.min(1).values.tolist(), ' 最大值:', p.max(1).values.tolist())
c = 5.0
print('加常数 c=5 后结果一致?', torch.allclose(softmax(x + c), p, atol=1e-6))
print('softmax 最大位置 == logits 最大位置?', (p.argmax(1) == x.argmax(1)).tolist())

每行和: [1.0, 1.0]
每行最小值: [0.039112575352191925, 0.005899750627577305]  最大值: [0.785597026348114, 0.8756006360054016]
加常数 c=5 后结果一致? True
softmax 最大位置 == logits 最大位置? [True, True]


## 小结与面试衔接

- softmax 三步：exp → 每行求和 → 归一化；输出是合法概率分布
- 数值稳定：减 max 不改变结果，但避免 exp 溢出（面试高频「softmax 为什么减 max」）
- 交叉熵 = 真实类概率的负对数；整批平均 = 负对数似然的平均（面试必问「为什么最小化交叉熵 = 极大似然」）
- 预告：为什么分类用交叉熵而回归用 MSE——交叉熵配合 softmax 梯度形式简洁、训练稳定；分类用 MSE 会梯度饱和难训练（面试对照表已列入）